<a href="https://colab.research.google.com/github/2303A51589/RL/blob/main/assi_8.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Creating `grid_env_det.py` file
We first write the deterministic grid world environment file so it can be imported as required by the assignment.

In [1]:
%%writefile grid_env_det.py
# Deterministic Grid World Environment definition
class GridEnv:
    def __init__(self):
        # States: S0, S1, S2 (terminal), S3, S4, S5 (terminal)
        # Actions: 'UP', 'DOWN', 'LEFT', 'RIGHT'
        self.states = ['S0', 'S1', 'S2', 'S3', 'S4', 'S5']
        self.terminal_states = {'S2', 'S5'}
        self.rewards = {
            ('S0', 'RIGHT'): ('S1', 0.0),
            ('S0', 'DOWN'): ('S3', 0.0),
            ('S1', 'LEFT'): ('S0', 0.0),
            ('S1', 'DOWN'): ('S4', 0.0),
            ('S1', 'RIGHT'): ('S2', 1.0), # S2 is terminal with +1 reward
            ('S3', 'UP'): ('S0', 0.0),
            ('S3', 'RIGHT'): ('S4', 0.0),
            ('S4', 'UP'): ('S1', 0.0),
            ('S4', 'LEFT'): ('S3', 0.0),
            ('S4', 'RIGHT'): ('S5', -1.0) # S5 is terminal with -1 reward
        }
        self.gamma = 0.9

    def step(self, state, action):
        if state in self.terminal_states:
            return state, 0.0, True

        key = (state, action.upper())
        if key in self.rewards:
            next_state, reward = self.rewards[key]
        else:
            # If an action leads to boundaries, the agent remains in the same state with 0 reward
            next_state, reward = state, 0.0

        done = next_state in self.terminal_states
        return next_state, reward, done

Writing grid_env_det.py


### Implementing the Assignment Code
Now we will load the module, define the policy, and construct both `state_value_function` and `action_value_function` using rollouts/trajectories to calculate the discounted return.

In [3]:
import grid_env_det

# Policy (Π) representation
policy = {
    'S0': 'RIGHT',
    'S1': 'DOWN',
    'S3': 'RIGHT',
    'S4': 'UP'
}

def state_value_function(env, start_state, policy, gamma=0.9):
    state = start_state
    discounted_return = 0.0
    discount = 1.0
    visited = set()

    # Follow the deterministic path until reaching a terminal state or detecting a loop
    while state not in env.terminal_states:
        if state in visited:
            # Loop detected (return is 0 for infinite loop with no reward)
            break
        visited.add(state)

        action = policy.get(state)
        if not action:
            break

        next_state, reward, done = env.step(state, action)
        discounted_return += discount * reward
        discount *= gamma
        state = next_state
        if done:
            break

    return discounted_return

def action_value_function(env, start_state, start_action, policy, gamma=0.9):
    # Take the first action manually as specified
    next_state, reward, done = env.step(start_state, start_action)
    discounted_return = reward

    # Then follow the policy Π from the next state onwards
    if not done:
        discounted_return += gamma * state_value_function(env, next_state, policy, gamma)

    return discounted_return

# Initialize environment
env = grid_env_det.GridEnv()

# Calculate state value for S0
s0_val = state_value_function(env, 'S0', policy, env.gamma)
print(f"State Value for state 'S0': {s0_val}")

# Calculate action value for S0 and action 'UP'
s0_action_up_val = action_value_function(env, 'S0', 'UP', policy, env.gamma)
print(f"Action Value for state 'S0' and action 'UP': {s0_action_up_val}")

State Value for state 'S0': 0.0
Action Value for state 'S0' and action 'UP': 0.0
